In [1]:
from finagg.fred.api import CategorySeries, SeriesObservations,SeriesCategories,Tags
import pandas as pd
so = SeriesObservations()
sc = SeriesCategories()
ret= []

def get_series(eventid):
    metadata = sc.get(eventid)
    eclass = metadata.iloc[-1]['name']
    df = so.get(eventid )[['series_id', 'value', 'date']]
    df.columns= ['event', 'actual', 'datetime']
    df['eclass'] = eclass
    df['datetime'] = pd.to_datetime(df.datetime)
    return df



In [2]:
from tqdm import tqdm

assets = [
    'WGCAL',
    'WOSDRL',
    'WACL',
    'WABPL',
    'WPCLC',
    'WFCDA',
    'WORAL',
    'WLCFLL',
    'H41RESPPAAENWW',# MS Facilities 2020 LLC (Wednesday level) -> verify this one on FRED before using    
    'SWPT',
    'WSHOBL',
    'WSHONBNL',
    'WSHONBIIL',
    'WSHOICL',
    'WSHOFADSL',
    'WSHOMCB',
    'WUDSHO',
    'WUPSHO',
    'WAOAL',
]

assets_name = [
    'Gold certificate account',
    'SDR certificate account',
    'Coin',
    'Bank premises',
    'Items in process of collection',
    'Foreign currency denominated assets',
    'Repurchase agreements',
    'Loans',
    'Net portfolio holdings of MS Facilities 2020 LLC (Main Street Lending Program)',
    'Central bank liquidity swaps',
    'Treasury bills',
    'Treasury notes and bonds (nominal)',
    'Treasury notes and bonds (inflation-indexed principal)',
    'Inflation compensation',
    'Federal agency debt securities',
    'Mortgage-backed securities',
    'Unamortized discounts',
    'Unamortized premiums',
    'Other assets',
]

liabilities = [
    'WLFN', 'WLRRAL', 'WDFOL', 'WDTGAL', 'WLODLL',
    'WLODL', 'WLDACLC', 'WLAD', 'TERMT', 'H41RESH4ENWW'
]

liabilities_name = [
    'Federal Reserve notes',
    'Reverse repurchase agreements',
    'Foreign official deposits',
    'Treasury General Account',
    'Other deposits - depository institutions',
    'Other deposits',
    'Deferred availability cash items',
    'Other liabilities and accrued dividends',
    'Term deposits',
    'Treasury Contribution to Credit Facilities',
]

import time
df_assets = []
for k, v in tqdm(zip(assets_name, assets)):
    try:
        df = get_series(v)
    except:
        time.sleep(100)
        df = get_series(v)
    df['ind'] = k
    df_assets.append(df)

df_asset = pd.concat(df_assets, ignore_index=True)


df_liabilities = []
for k, v in tqdm(zip(liabilities_name, liabilities)):
    try:
        df = get_series(v)
    except:
        time.sleep(100)
        df = get_series(v)
    df['ind'] = k
    df_liabilities.append(df)

df_liability = pd.concat(df_liabilities, ignore_index=True)



19it [00:03,  5.87it/s] 
10it [01:49, 10.93s/it]


In [ ]:
df_liability

In [3]:
import plotly.express as px

liability_wide = (
    df_liability
    .pivot(index='datetime', columns='ind', values='actual')
    .sort_index()
    .reset_index()
)

fig = px.area(
    liability_wide,
    x='datetime',
    y=[c for c in liability_wide.columns if c != 'datetime'],
    title='Fed Liability'
)

fig.update_yaxes(title='USD million')
fig.update_xaxes(title='')
fig.show()


In [14]:
import plotly.express as px

asset_wide = (
    df_asset
    .pivot(index='datetime', columns='ind', values='actual')
    .sort_index()
    .reset_index()
)

fig = px.area(
    asset_wide,
    x='datetime',
    y=[c for c in asset_wide.columns if c != 'datetime'],
    title='Fed Assets'
)

fig.update_xaxes(
    title='',
    # dtick=7 * 24 * 60 * 60 * 1000,
    tickformat='%Y-%m-%d',
    tickangle=45
)

fig.update_yaxes(title='USD million')
# fig.update_xaxes(title='')
fig.show()